In [ ]:
!pip install ctgan sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.0/197.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.3/198.3 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from ctgan import CTGAN
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer
import joblib
import kagglehub
import os

In [ ]:
print("Mengunduh dataset dari KaggleHub...")
path = kagglehub.dataset_download("stephanmatzka/predictive-maintenance-dataset-ai4i-2020")

csv_file_path = os.path.join(path, 'ai4i2020.csv')

df = pd.read_csv(csv_file_path)
print(f"Data berhasil dimuat. Jumlah baris: {len(df)}")

Mengunduh dataset dari KaggleHub...


100%|██████████| 136k/136k [00:00<00:00, 25.6MB/s]

Extracting files...
Data berhasil dimuat. Jumlah baris: 10000


In [ ]:
df = df[df['RNF'] != 1]
df = df[df['RNF'] != 1].reset_index(drop=True)
df = df.drop(columns=['UDI', 'RNF'])
print(f"Jumlah data setelah hapus Random Failures: {len(df)}")
print("✅ Random Failures berhasil dihapus agar model fokus pada pola fisik.")

Jumlah data setelah hapus Random Failures: 9981
✅ Random Failures berhasil dihapus agar model fokus pada pola fisik.


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9981 entries, 0 to 9980
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product ID               9981 non-null   object 
 1   Type                     9981 non-null   object 
 2   Air temperature [K]      9981 non-null   float64
 3   Process temperature [K]  9981 non-null   float64
 4   Rotational speed [rpm]   9981 non-null   int64  
 5   Torque [Nm]              9981 non-null   float64
 6   Tool wear [min]          9981 non-null   int64  
 7   Machine failure          9981 non-null   int64  
 8   TWF                      9981 non-null   int64  
 9   HDF                      9981 non-null   int64  
 10  PWF                      9981 non-null   int64  
 11  OSF                      9981 non-null   int64  
dtypes: float64(3), int64(7), object(2)
memory usage: 935.8+ KB


In [ ]:
real_data = df.drop(columns=[ 'Product ID'])

metadata = Metadata()
metadata.detect_from_dataframe(data=real_data, table_name='maintenance_data')

print("🤖 Sedang melatih 'Otak' Generative AI (CTGAN)...")
print("Ini akan memakan waktu beberapa menit karena menggunakan Deep Learning...")

conditions = [
    (df['TWF'] == 1),
    (df['HDF'] == 1),
    (df['PWF'] == 1),
    (df['OSF'] == 1)
]

choices = [
    'Tool Wear Failure',
    'Heat Dissipation Failure',
    'Power Failure',
    'Overstrain Failure'
]

# Buat kolom sementara 'Failure Type'
df['Failure Type'] = np.select(conditions, choices, default='No Failure')

# Hapus kolom yang tidak relevan untuk pembelajaran pola fisika
# Drop kolom binary asli sementara waktu agar CTGAN fokus ke fisika (suhu/speed), bukan label.
cols_to_drop = [ 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF']
real_data = df.drop(columns=cols_to_drop)

print("📊 Struktur data untuk training AI:")
print(real_data.head())

metadata = Metadata()
metadata.detect_from_dataframe(data=real_data, table_name='maintenance_data')

print("\n🤖 Sedang melatih 'Otak' Generative AI (CTGAN)...")

failure_types = ['Power Failure', 'Tool Wear Failure', 'Overstrain Failure', 'Heat Dissipation Failure']
synthetic_data_storage = []

🤖 Sedang melatih 'Otak' Generative AI (CTGAN)...
Ini akan memakan waktu beberapa menit karena menggunakan Deep Learning...
📊 Struktur data untuk training AI:
  Product ID Type  Air temperature [K]  Process temperature [K]  \
0     M14860    M                298.1                    308.6   
1     L47181    L                298.2                    308.7   
2     L47182    L                298.1                    308.5   
3     L47183    L                298.2                    308.6   
4     L47184    L                298.2                    308.7   

   Rotational speed [rpm]  Torque [Nm]  Tool wear [min] Failure Type  
0                    1551         42.8                0   No Failure  
1                    1408         46.3                3   No Failure  
2                    1498         49.4                5   No Failure  
3                    1433         39.5                7   No Failure  
4                    1408         40.0                9   No Failure  

🤖 Sedang mel

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9981 entries, 0 to 9980
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product ID               9981 non-null   object 
 1   Type                     9981 non-null   object 
 2   Air temperature [K]      9981 non-null   float64
 3   Process temperature [K]  9981 non-null   float64
 4   Rotational speed [rpm]   9981 non-null   int64  
 5   Torque [Nm]              9981 non-null   float64
 6   Tool wear [min]          9981 non-null   int64  
 7   Machine failure          9981 non-null   int64  
 8   TWF                      9981 non-null   int64  
 9   HDF                      9981 non-null   int64  
 10  PWF                      9981 non-null   int64  
 11  OSF                      9981 non-null   int64  
 12  Failure Type             9981 non-null   object 
dtypes: float64(3), int64(7), object(3)
memory usage: 1013.8+ KB


In [ ]:
from sdv.metadata import SingleTableMetadata

df_normal = real_data[real_data['Failure Type'] == 'No Failure'].sample(1000)

for fail_type in failure_types:
    print(f"\n⚡ Mempelajari pola kerusakan: {fail_type}")

    # Filter data berdasarkan kolom 'Failure Type' yang telah dibuat
    df_fail = real_data[real_data['Failure Type'] == fail_type]

    # Cek jumlah data
    if len(df_fail) < 10:
        print(f"⚠️ Data terlalu sedikit ({len(df_fail)}), menggunakan Noise Injection.")
        augmented = df_fail.copy()
        for _ in range(20):
            noise = df_fail.copy()
            numeric_cols = ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
            for col in numeric_cols:
                noise[col] = noise[col] + np.random.normal(0, noise[col].std()*0.02, len(noise))
            augmented = pd.concat([augmented, noise])
        synthetic_data_storage.append(augmented)
        continue

    # Gabungkan dengan data normal untuk konteks (Contrastive Training)
    training_subset = pd.concat([df_fail, df_fail])

    training_subset_clean = training_subset.drop(columns=['Product ID'])

    temp_metadata = SingleTableMetadata()
    temp_metadata.detect_from_dataframe(training_subset_clean)

    if 'Type' in training_subset_clean.columns:
        temp_metadata.update_column(column_name='Type', sdtype='categorical')
    if 'Failure Type' in training_subset_clean.columns:
        temp_metadata.update_column(column_name='Failure Type', sdtype='categorical')

    # Training CTGAN
    synthesizer = CTGANSynthesizer(temp_metadata, epochs=300, verbose=False)
    synthesizer.fit(training_subset_clean)

    # Generate Data Baru
    print(f"   Generating data sintetik valid untuk {fail_type}...")
    synthetic = synthesizer.sample(num_rows=7000)
    synthetic['Failure Type'] = fail_type

    # Validasi Perhitungan Kondisi
    power = 2 * np.pi * (synthetic['Rotational speed [rpm]'] / 60) * synthetic['Torque [Nm]']
    valid_rows = []

    for index, row in synthetic.iterrows():
        is_valid = True

        # Validasi PWF
        if fail_type == 'Power Failure':
            if 3500 <= power[index] <= 9000: is_valid = False

        # Validasi HDF
        elif fail_type == 'Heat Dissipation Failure':
            diff = row['Process temperature [K]'] - row['Air temperature [K]']
            if diff > 8.6 or row['Rotational speed [rpm]'] > 1380:
                row['Rotational speed [rpm]'] = np.random.randint(1100, 1370)

        # Validasi OSF
        elif fail_type == 'Overstrain Failure':
            limit = 11000 if row['Type'] == 'L' else (12000 if row['Type'] == 'M' else 13000)
            if (row['Tool wear [min]'] * row['Torque [Nm]']) < limit:
                row['Torque [Nm]'] = (limit / row['Tool wear [min]']) + 2

        # Validasi TWF
        elif fail_type == 'Tool Wear Failure':
             if row['Tool wear [min]'] < 200:
                 row['Tool wear [min]'] = np.random.randint(200, 240)

        if is_valid:
            valid_rows.append(row)

    if valid_rows:
        synthetic_clean = pd.DataFrame(valid_rows)
        synthetic_data_storage.append(synthetic_clean)

# Penggabungan & Rekonstruksi Format Asli
print("\n📦 Menggabungkan data asli dan sintetik...")
df_synthetic_final = pd.concat(synthetic_data_storage)

# Gabung dengan data asli (tapi drop kolom binary asli dulu biar seragam kolomnya)
df_original_clean = df.drop(columns=['Product ID', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF'])
df_final = pd.concat([df_original_clean, df_synthetic_final])
df_final = df_final.reset_index(drop=True)

# Rekonstruksi Kolom
# Kembalikan kolom TWF, HDF, dll berdasarkan 'Failure Type'
print("🔄 Mengembalikan format kolom binary (TWF, HDF, dll)...")

df_final['TWF'] = np.where(df_final['Failure Type'] == 'Tool Wear Failure', 1, 0)
df_final['HDF'] = np.where(df_final['Failure Type'] == 'Heat Dissipation Failure', 1, 0)
df_final['PWF'] = np.where(df_final['Failure Type'] == 'Power Failure', 1, 0)
df_final['OSF'] = np.where(df_final['Failure Type'] == 'Overstrain Failure', 1, 0)

# Kolom Machine Failure (Target Umum)
df_final['Machine failure'] = np.where(df_final['Failure Type'] != 'No Failure', 1, 0)

# Tambahkan placeholder untuk kolom yang tadi didrop
df_final['UDI'] = range(1, len(df_final) + 1)
df_final['Product ID'] = 'SYNTHETIC' # Tandai data buatan

output_file = 'predictive_maintenance_augmented_ctgan.csv'
df_final.to_csv(output_file, index=False)

print(f"✅ Selesai! Dataset baru tersimpan di '{output_file}'")
print(f"Total Data: {len(df_final)}")
print("Distribusi Kelas Baru:")
print(df_final['Failure Type'].value_counts())


⚡ Mempelajari pola kerusakan: Power Failure


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


   Generating data sintetik valid untuk Power Failure...

⚡ Mempelajari pola kerusakan: Tool Wear Failure


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


   Generating data sintetik valid untuk Tool Wear Failure...

⚡ Mempelajari pola kerusakan: Overstrain Failure


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


   Generating data sintetik valid untuk Overstrain Failure...

⚡ Mempelajari pola kerusakan: Heat Dissipation Failure


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(


   Generating data sintetik valid untuk Heat Dissipation Failure...

📦 Menggabungkan data asli dan sintetik...
🔄 Mengembalikan format kolom binary (TWF, HDF, dll)...
✅ Selesai! Dataset baru tersimpan di 'predictive_maintenance_augmented_ctgan.csv'
Total Data: 35874
Distribusi Kelas Baru:
Failure Type
No Failure                  9652
Heat Dissipation Failure    7115
Overstrain Failure          7078
Tool Wear Failure           7045
Power Failure               4984
Name: count, dtype: int64
